# Swin2SR classical-sr-x2-64 — DIMER image super-resolution tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/swin2sr-super-resolution-pipeline/blob/main/tutorials/swin2sr_super_resolution_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-caidas%2Fswin2SR--classical--sr--x2--64-ffcc4d?style=flat)](https://huggingface.co/caidas/swin2SR-classical-sr-x2-64)
[![Upstream](https://img.shields.io/badge/Upstream-mv--lab%2Fswin2sr-181717?style=flat&logo=github&logoColor=white)](https://github.com/mv-lab/swin2sr)
[![arXiv](https://img.shields.io/badge/arXiv-2209.11345-b31b1b.svg)](https://arxiv.org/abs/2209.11345)

**Profile:** `TASK-INFERENCE`
**Notebook specification:** DIMER Notebook Specification 1.0
**Capability:** 2× single-image super-resolution (classical SR) using the pinned `caidas/swin2SR-classical-sr-x2-64` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. At inference the image is rescaled to `[0, 1]` and padded to a multiple of 8, a SwinV2 transformer with `patch_size` 1 (every pixel is a token) runs windowed attention over the whole input, and the pipeline crops the padding off and returns a uint8 RGB array with exactly twice the input width and height. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and the processor configuration, and this repository adds packaging, snapshot verification, input validation with named ceilings, a fixed output contract and the `psnr` helper. The default sample is a synthetic image generated in code; the PSNR reported for it is sanity evidence against a self-made reference, not a benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, generate a synthetic high-resolution image and derive its low-resolution input in code, validate the input against the pipeline's side ceilings, run 2× super-resolution through the public API, compute `psnr` against the self-made reference and a bicubic baseline and read both as sanity evidence, exercise an optional BYOD path, and export the upscaled image plus machine-readable provenance.

**This notebook does not demonstrate:** scales other than 2×, compressed-image or real-world (blind) restoration, denoising, JPEG-artifact removal, video super-resolution, benchmark evaluation on Set5/Set14/DIV2K, or any training. A PSNR against a reference you downscaled yourself measures how well the model inverts *that* downscaling kernel, nothing more.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate for the default 64×48 input: the repository's model card records 0.36 s per `upscale` on that size in the Windows venv (Intel Core Ultra 9 275HX), 7.3 s for 256×256 and 34 s for 512×512. Cost grows with input area because every pixel is an attention token; the pinned `torch==2.14.0` install is the largest download of the run.
- **Knowledge:** basic Python, NumPy and PIL image handling; what PSNR measures (pixel-wise fidelity, in dB, higher is closer).
- **Data:** the default sample is a deterministic 128×96 RGB image generated in code (gradient background, a filled square and a diagonal stripe), bicubic-downscaled to 64×48 as the model input; nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, both sides between 8 and 512 px; a BYOD image has no reference, so no PSNR is computed for it. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** GitHub (repository clone) and the Hugging Face Hub (the package's `stage_missing_files` fetches the pinned checkpoint, ~48 MB, once, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `safetensors`, `numpy`, `pillow`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported, the cell fails with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the repository revision, Python, `torch` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline.git'
REPO_NAME = 'swin2sr-super-resolution-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic** and carries its own reference: a deterministic 128×96 RGB image is built in code (horizontal colour gradient, one filled square, one diagonal stripe — sharp edges are what a super-resolver has to reconstruct), kept as the **high-resolution reference**, and bicubic-downscaled to 64×48 to become the model input. Both digests are printed. This is the same kind of gradient-and-square input the repository's smoke run used. Because you made the reference yourself, PSNR against it later is a self-consistency check of the input contract and forward pass, not a benchmark: real benchmarks use fixed public HR/LR pairs and a specified degradation kernel. BYOD is optional and disabled by default; when enabled, upload one image and it is upscaled as-is with no reference.

Before anything expensive runs, this cell surfaces the pipeline's operational ceilings — `UPSCALE` (2), `MIN_INPUT_SIDE` (8 px; the processor pads to a multiple of the 8-px attention window) and `MAX_INPUT_SIDE` (512 px) — and checks the input against them with a clear message. The 512 px ceiling was lowered from 1024 px by the repository owner on 2026-09-12 because the card-pass smoke measured 160 s for a 1024×1024 input on CPU against 34 s for 512×512, and DIMER validators must stay responsive; larger images must be tiled by the caller. Inside the pipeline the image is converted to RGB and padded to a multiple of 8, and the padding is cropped off at output scale; nothing else is dropped or altered. Look for a dictionary naming the sample kind, the input and reference sizes and digests.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

from swin2sr_super_resolution_pipeline import MAX_INPUT_SIDE, MIN_INPUT_SIDE, UPSCALE

USE_BYOD = False  # @param {type:"boolean"}
HR_WIDTH, HR_HEIGHT = 128, 96

print({'ceilings': {'UPSCALE': UPSCALE, 'MIN_INPUT_SIDE': MIN_INPUT_SIDE, 'MAX_INPUT_SIDE': MAX_INPUT_SIDE}})
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    lr_image = Image.open(io.BytesIO(uploaded[image_name]))
    lr_image.load()
    hr_reference = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic high-resolution reference: no randomness, so no seed is needed.
    ramp = np.linspace(0.0, 255.0, HR_WIDTH)
    red = np.tile(ramp, (HR_HEIGHT, 1))
    green = np.tile(np.linspace(255.0, 0.0, HR_HEIGHT), (HR_WIDTH, 1)).T
    blue = np.full((HR_HEIGHT, HR_WIDTH), 96.0)
    hr_reference = Image.fromarray(np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8), mode='RGB')
    draw = ImageDraw.Draw(hr_reference)
    draw.rectangle([24, 20, 60, 56], fill=(20, 20, 20))
    draw.line([(70, 84), (118, 12)], fill=(250, 250, 250), width=5)
    # The model input is the reference bicubic-downscaled by exactly UPSCALE; the kernel choice is part of the sample.
    lr_image = hr_reference.resize((HR_WIDTH // UPSCALE, HR_HEIGHT // UPSCALE), Image.Resampling.BICUBIC)
    image_name = f'synthetic_shapes_{HR_WIDTH // UPSCALE}x{HR_HEIGHT // UPSCALE}.png'
    sample_kind = 'synthetic'

width, height = lr_image.size
if min(width, height) < MIN_INPUT_SIDE or max(width, height) > MAX_INPUT_SIDE:
    raise ValueError(f'{image_name}: input {lr_image.size} must have both sides within {MIN_INPUT_SIDE}..{MAX_INPUT_SIDE} px; resize or tile the image and rerun this cell.')
lr_sha256 = hashlib.sha256(np.asarray(lr_image.convert('RGB')).tobytes()).hexdigest()
hr_sha256 = None if hr_reference is None else hashlib.sha256(np.asarray(hr_reference).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'input_size': lr_image.size, 'input_rgb_sha256': lr_sha256, 'reference_size': None if hr_reference is None else hr_reference.size, 'reference_rgb_sha256': hr_sha256})

## 3. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository carries `weights/swin2sr-x2-64/dimer-base-manifest.json`, `config.json` and `preprocessor_config.json` but git-ignores the 48 MB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, at the pinned revision, into the snapshot directory — it prints the list it fetched (`[]` on a warm runtime) and refuses a manifest whose identity differs from the package pins. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every manifest entry (size and SHA-256) and raises on the first mismatch; its returned dict is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files with `local_files_only=True` and `trust_remote_code=False` — there is no fallback to a different download. The effective model identity and the device chosen (`cuda:0` when available, else `cpu`) are printed before inference.

In [ ]:
from swin2sr_super_resolution_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, Swin2SRPipeline, psnr, stage_missing_files, verify_snapshot
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'upscale': UPSCALE})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print(snapshot)
pipe = Swin2SRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device})

## 4. Upscale and read the fidelity numbers correctly

`upscale` returns a dict with `image` (uint8 array of shape `(2H, 2W, 3)`), `scale` (2), `input_size` and `output_size` as `(width, height)`, and the model identity. The output is a deterministic reconstruction — no sampling, no seed, `torch.inference_mode` — so repeated runs on the same device and dtype give the same bytes; CPU versus CUDA kernels can differ in the last rounding step.

The repository's only metric helper is `psnr(pred, ref)`: peak signal-to-noise ratio in dB between two uint8 RGB arrays of identical shape (`10·log10(255² / MSE)`, `inf` when identical). It is computed only when a high-resolution reference exists — on the synthetic default path, against the reference you generated in Section 2; on BYOD, none exists and none is reported. As a **meaningful baseline** the same PSNR is computed for a plain bicubic 2× resize of the input, so you can see whether the model beats interpolation on this one image. Both numbers are single-image tutorial evidence with no dispersion estimate; they depend on the downscaling kernel you chose and say nothing about photographs, compression artefacts, or the Set5/Set14/DIV2K figures reported by the upstream paper (not measured here). A PSNR against a benchmark needs public HR/LR pairs with the benchmark's own degradation. Look for `output_size` equal to twice `input_size`, and a model PSNR above the bicubic baseline on the sharp-edged synthetic image.

In [ ]:
result = pipe.upscale(lr_image)
sr_array = result['image']
print({'scale': result['scale'], 'input_size': result['input_size'], 'output_size': result['output_size'], 'array_shape': sr_array.shape, 'dtype': str(sr_array.dtype), 'device': pipe.device})
metrics = {}
if hr_reference is not None:
    hr_array = np.asarray(hr_reference)
    bicubic_array = np.asarray(lr_image.convert('RGB').resize(hr_reference.size, Image.Resampling.BICUBIC))
    metrics['psnr_db'] = {'model_vs_self_made_reference': psnr(sr_array, hr_array), 'bicubic_baseline_vs_self_made_reference': psnr(bicubic_array, hr_array)}
    print({'sample_metrics': metrics, 'note': 'single synthetic image, self-made reference; sanity evidence, not a benchmark'})
else:
    print('No high-resolution reference exists for a BYOD image, so psnr is not computed; inspect the exported PNG instead.')

## 5. Export outputs and provenance

The upscaled image is written as PNG (`outputs/swin2sr_super_resolution_output.png`) — the actual artifact a downstream consumer wants — and machine-readable JSON preserves the sizes, the sample identity and digests, the sanity PSNR values when computed, the repository revision, the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `transformers`, device). No credentials are recorded.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
Image.fromarray(sr_array, mode='RGB').save('outputs/swin2sr_super_resolution_output.png')
payload = {
    'prediction': {key: value for key, value in result.items() if key != 'image'},
    'output_file': 'outputs/swin2sr_super_resolution_output.png',
    'output_rgb_sha256': hashlib.sha256(sr_array.tobytes()).hexdigest(),
    'metrics': metrics,
    'sample': {'kind': sample_kind, 'name': image_name, 'input_size': list(lr_image.size), 'input_rgb_sha256': lr_sha256, 'reference_rgb_sha256': hr_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/swin2sr_super_resolution_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The upscaled image is a learned reconstruction: plausible detail synthesised from the low-resolution input, not recovered ground truth. The PSNR values shown on the synthetic path compare the model and a bicubic baseline against a reference you generated and downscaled yourself, so they measure how well each inverts *that* bicubic downscaling on one synthetic image; they are not comparable to published Set5/Set14/DIV2K numbers and must not be generalised to photographs, other kernels, compressed inputs, or other scales. Inputs above 512 px per side are refused (tile them), the model handles only 2×, and content that was never in the input cannot be recovered. The pipeline provides no denoising, artefact removal, blind restoration, video, or training capability.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a small photograph (≤ 512 px per side) and inspect the exported PNG next to a bicubic resize; regenerate the synthetic reference with `Image.Resampling.NEAREST` or `BOX` downscaling and watch how much the model PSNR moves with the kernel alone; time `upscale` on 128×128, 256×256 and 512×512 inputs on your runtime to see the area scaling that motivated the ceiling.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/caidas/swin2SR-classical-sr-x2-64
- Upstream code: https://github.com/mv-lab/swin2sr
- Swin2SR: SwinV2 Transformer for Compressed Image Super-Resolution and Restoration (Conde et al., 2022): https://arxiv.org/abs/2209.11345